# SVD Collaborative Filtering — Google Colab Training
**Model:** Surprise SVD (Matrix Factorization)  
**Dataset:** MovieLens 32M — `train_ratings.csv` (31.8M ratings)  
**Output:** `svd_model.pkl` — download and place in `artifacts/collaborative/`

---
### Two ways to load your data (pick one):
| Option | How | Best for |
|--------|-----|----------|
| **A — Direct Upload** | Run Cell 2A → file picker appears | Fast internet, one-time run |
| **B — Google Drive** | Run Cell 2B → mount Drive | Slow internet, repeated runs |

> ⚠️ **Direct upload tip:** Colab disconnects after ~90 min idle. If it disconnects mid-training you'll need to re-upload. Use the keep-alive cell (Cell 3) to prevent this.

In [ ]:
!pip install scikit-surprise psutil -q
print('✅ Dependencies installed')

In [ ]:
# ══════════════════════════════════════════════════════
# OPTION A — Direct Upload (recommended if fast internet)
# A file picker will appear — select train_ratings.csv
# ══════════════════════════════════════════════════════
from google.colab import files
from pathlib import Path
import os

print('📂 File picker opening — select train_ratings.csv from your Mac...')
print('   (File is ~1.5 GB — upload time depends on your internet speed)\n')

uploaded = files.upload()   # ← file picker appears here

# After upload, find the file in /content/
TRAIN_RATINGS_PATH = None
for filename in uploaded.keys():
    if 'rating' in filename.lower():
        TRAIN_RATINGS_PATH = f'/content/{filename}'
        break

if not TRAIN_RATINGS_PATH:
    TRAIN_RATINGS_PATH = f'/content/{list(uploaded.keys())[0]}'

size_gb = Path(TRAIN_RATINGS_PATH).stat().st_size / 1e9
print(f'✅ Uploaded: {TRAIN_RATINGS_PATH}')
print(f'   Size: {size_gb:.2f} GB')
print('\n⚠️  Run the keep-alive cell (next) before starting training!')

In [ ]:
# ══════════════════════════════════════════════════════
# OPTION B — Google Drive (skip this if you used Option A)
# Upload train_ratings.csv to My Drive first, then run this
# ══════════════════════════════════════════════════════
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

TRAIN_RATINGS_PATH = None
print('Searching Google Drive for train_ratings.csv...')
for root, _, files_list in os.walk('/content/drive/My Drive'):
    for f in files_list:
        if f == 'train_ratings.csv':
            TRAIN_RATINGS_PATH = os.path.join(root, f)
            break
    if TRAIN_RATINGS_PATH:
        break

if not TRAIN_RATINGS_PATH:
    raise FileNotFoundError('train_ratings.csv not found in Google Drive. Upload it first.')

print(f'✅ Found: {TRAIN_RATINGS_PATH}')
print(f'   Size:  {Path(TRAIN_RATINGS_PATH).stat().st_size / 1e9:.2f} GB')

In [ ]:
# ══════════════════════════════════════════════════════
# KEEP-ALIVE — Run this before training!
# Prevents Colab from disconnecting during the ~45 min training
# This cell runs a background thread that clicks "stay connected"
# ══════════════════════════════════════════════════════
import threading
import time

def keep_alive():
    """Prints a heartbeat every 5 minutes to prevent Colab idle timeout"""
    count = 0
    while True:
        time.sleep(300)  # every 5 minutes
        count += 1
        print(f'  💓 Keep-alive heartbeat #{count} ({count*5} min elapsed)')

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print('✅ Keep-alive thread started — Colab will stay connected during training')
print('   (Heartbeat prints every 5 minutes in training cell output)')

In [ ]:
import psutil
import os

ram_gb   = psutil.virtual_memory().total / 1e9
ram_free = psutil.virtual_memory().available / 1e9
cpu_count = os.cpu_count()

print('── Colab Environment ──────────────────────')
print(f'  Total RAM : {ram_gb:.1f} GB')
print(f'  Free RAM  : {ram_free:.1f} GB')
print(f'  CPU cores : {cpu_count}')
print('  Note: Surprise SVD uses 1 core (single-threaded)')
print('───────────────────────────────────────────')

if ram_free < 6:
    print('\n⚠️  Low free RAM! Go to Runtime → Restart Runtime to clear memory.')
else:
    print('\n✅ Sufficient RAM for training.')

In [ ]:
import pandas as pd
import time
import psutil

print(f'Loading {TRAIN_RATINGS_PATH}...')
print('Reading in 5M-row chunks to keep RAM stable...\n')

t0     = time.time()
chunks = []

for chunk in pd.read_csv(
    TRAIN_RATINGS_PATH,
    usecols=['userId', 'movieId', 'rating'],   # skip timestamp — not needed for SVD
    dtype={'userId': 'int32', 'movieId': 'int32', 'rating': 'float32'},
    chunksize=5_000_000
):
    chunks.append(chunk)
    total_so_far = sum(len(c) for c in chunks)
    ram_used = psutil.virtual_memory().used / 1e9
    print(f'  Chunk {len(chunks):02d}: {total_so_far:>12,} rows loaded  |  RAM: {ram_used:.1f} GB')

ratings = pd.concat(chunks, ignore_index=True)
del chunks

print(f'\n✅ Fully loaded in {time.time()-t0:.1f}s')
print(f'   Total ratings : {len(ratings):,}')
print(f'   Unique users  : {ratings["userId"].nunique():,}')
print(f'   Unique movies : {ratings["movieId"].nunique():,}')
print(f'   RAM used now  : {psutil.virtual_memory().used / 1e9:.1f} GB')

In [ ]:
from surprise import Dataset, Reader, SVD
import time
import psutil

# ── CONFIG (change if needed) ────────────────────────
N_FACTORS = 20   # latent factors — higher = more expressive, slower
N_EPOCHS  = 15   # SGD passes — more = better accuracy, slower
# ─────────────────────────────────────────────────────

print('Step 1/3 — Building Surprise trainset...')
t0 = time.time()
reader   = Reader(rating_scale=(0.5, 5.0))
data     = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset = data.build_full_trainset()
print(f'  ✅ Done in {time.time()-t0:.1f}s')
print(f'     Ratings : {trainset.n_ratings:,}')
print(f'     Users   : {trainset.n_users:,}')
print(f'     Movies  : {trainset.n_items:,}')

# Free DataFrame — Surprise has its own internal copy now
del ratings
print(f'     RAM after cleanup: {psutil.virtual_memory().used / 1e9:.1f} GB')

print(f'\nStep 2/3 — Training SVD (factors={N_FACTORS}, epochs={N_EPOCHS})...')
print('Estimated time: 40–50 minutes. Each epoch line below = one pass.\n')

model   = SVD(n_factors=N_FACTORS, n_epochs=N_EPOCHS, random_state=42, verbose=True)
t_train = time.time()
model.fit(trainset)
elapsed = time.time() - t_train

print(f'\nStep 3/3 — Done!')
print(f'  ✅ Training time: {elapsed/60:.1f} minutes')

In [ ]:
import joblib
from pathlib import Path
from google.colab import files

SAVE_PATH = '/content/svd_model.pkl'

# Quick sanity check
test_pred = model.predict(uid=1, iid=1)
print(f'Sanity check — User 1 + Movie 1: predicted rating = {test_pred.est:.2f}')
assert 0.5 <= test_pred.est <= 5.0, 'Prediction out of range — something is wrong!'
print('✅ Sanity check passed\n')

# Save
print(f'Saving to {SAVE_PATH}...')
joblib.dump(model, SAVE_PATH, compress=3)
size_mb = Path(SAVE_PATH).stat().st_size / 1e6
print(f'✅ Saved — {size_mb:.1f} MB\n')

# Auto-download
print('Starting download...')
files.download(SAVE_PATH)

print()
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('AFTER DOWNLOADING:')
print('  Move svd_model.pkl to:')
print('  artifacts/collaborative/svd_model.pkl')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')